# Create rdf

In [15]:
from pykeen.cli import datasets
from rdflib import Graph, URIRef, BNode, RDF, RDFS
from rdflib.namespace import OWL
from rdflib.collection import Collection
from itertools import combinations
from collections import defaultdict
import pandas as pd
from pykeen import datasets
from pykeen.triples import TriplesFactory
import numpy as np


In [18]:
base_iri = 'http://example.org/'

hn = Graph()

defined_classes = set()
property_definitions = defaultdict(lambda: {'domains': set(), 'ranges': set()})
property_map = dict()
tbox = pd.read_csv('metaedges.tsv', sep='\t')
for i,row in tbox.iterrows():
    prop_line = row['metaedge']

    if '-' in prop_line:
        prop_name = row['metaedge'].replace(' ','').split('-')[1]
        domain_str, prop_str, range_str = [item.replace(' ','') for item in prop_line.split(' - ')]

    else :
        prop_name= row['metaedge'].replace(' ','').split('>')[1]
        domain_str, prop_str, range_str = [item.replace(' ','') for item in prop_line.split(' > ')]

    property_map[row['abbreviation']] = prop_name
    property_definitions[prop_str]['domains'].add(domain_str)
    property_definitions[prop_str]['ranges'].add(range_str) # Also collect ranges for completeness

    if domain_str not in defined_classes:
        hn.add((URIRef(base_iri + domain_str.replace(' ','')), RDF.type, RDFS.Class))
        defined_classes.add(domain_str)
    if range_str not in defined_classes:
        hn.add((URIRef(base_iri + range_str.replace(' ','')), RDF.type, RDFS.Class))
        defined_classes.add(range_str)

for class_name in defined_classes:
    hn.add((URIRef(base_iri + class_name), RDF.type, RDFS.Class))

for prop_str, defs in property_definitions.items():
    prop_uri = URIRef(base_iri + prop_str.replace(' ',''))
    hn.add((prop_uri, RDF.type, RDF.Property))

    domain_uris = [URIRef(base_iri +d.replace(' ','')) for d in defs['domains']]
    if len(domain_uris) == 1:
        # Simple case: only one domain
        hn.add((prop_uri, RDFS.domain, domain_uris[0]))
    else:
        # Complex case: create a union class for the domain
        union_bnode = BNode() # An anonymous class for the union
        hn.add((union_bnode, RDF.type, RDFS.Class))
        # Use rdflib's Collection to create the RDF list for owl:unionOf
        c = Collection(hn, BNode(), domain_uris)
        hn.add((union_bnode, OWL.unionOf, c.uri))
        hn.add((prop_uri, RDFS.domain, union_bnode))

    range_uris = [URIRef(base_iri + r.replace(' ','')) for r in defs['ranges']]
    if len(range_uris) == 1:
        hn.add((prop_uri, RDFS.range, range_uris[0]))
    else:
        union_bnode = BNode()
        hn.add((union_bnode, RDF.type, RDFS.Class))
        c = Collection(hn, BNode(), range_uris)
        hn.add((union_bnode, OWL.unionOf, c.uri))
        hn.add((prop_uri, RDFS.range, union_bnode))
class_uris = [URIRef(base_iri + c.replace(' ','')) for c in defined_classes]

for class1 in class_uris:
    for class2 in class_uris:
        if class1 != class2:
            hn.add((class1, OWL.disjointWith, class2))
print(len(hn))
hn.serialize(format='nt', destination='hetionet/hetionet_tbox.nt')


207


/Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


<Graph identifier=N48e57be5e221473da2a88e60d3e2dfff (<class 'rdflib.graph.Graph'>)>

In [19]:
# add individuals and types
with open('entity_mapping.nt','r') as f:

    for line in f.readlines():
        names = line.replace('<','').replace('>','')
        s,p,o,_ = names.split(' ')
        hn.add((URIRef(base_iri + s), RDF.type ,URIRef(base_iri + o)))

print(len(hn))

45365


In [10]:
# hetio = datasets.Hetionet(42)
# new_hetio = datasets.Dataset().from_tf(tf=hetio.merged(), ratios=[0.98,.01,0.01])
# np.savetxt('hetio_train.tsv', new_hetio.training.label_triples(new_hetio.training.mapped_triples), delimiter='\t', fmt='%s')
# np.savetxt('hetio_test.tsv', new_hetio.testing.label_triples(new_hetio.testing.mapped_triples), delimiter='\t', fmt='%s')
# np.savetxt('hetio_validation.tsv', new_hetio.validation.label_triples(new_hetio.validation.mapped_triples), delimiter='\t', fmt='%s')

In [20]:
# add individual relations from train

with open('hetio_train.tsv', 'r') as f:
    cnt = 0
    for line in f.readlines():
        cnt += 1
        s,p,o = line.strip().split('\t')
        hn.add((URIRef(base_iri + s.split('::')[-1]), URIRef(base_iri + property_map.get(p)) ,URIRef(base_iri + o.split('::')[-1])))

hn.serialize('hetionet/hetio_train_graph.nt', format='nt')

/Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


<Graph identifier=N48e57be5e221473da2a88e60d3e2dfff (<class 'rdflib.graph.Graph'>)>

In [21]:
with open('hetio_validation.tsv','r') as f:
    cnt = 0
    for line in f.readlines():
        cnt += 1
        s,p,o = line.strip().split('\t')
        hn.add((URIRef(base_iri +s.split('::')[-1]), URIRef(base_iri +property_map.get(p)) ,URIRef(base_iri +o.split('::')[-1])))
with open('hetio_test.tsv', 'r') as f:
    cnt = 0
    for line in f.readlines():
        cnt += 1
        s, p, o = line.strip().split('\t')
        hn.add((URIRef(base_iri +s.split('::')[-1]), URIRef(base_iri +property_map.get(p)), URIRef(base_iri +o.split('::')[-1])))

hn.serialize('hetionet/hetio_full_graph.nt', format='nt')

/Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


<Graph identifier=N48e57be5e221473da2a88e60d3e2dfff (<class 'rdflib.graph.Graph'>)>

In [22]:
# make nicer datasets
for df_name in ['hetio_train.tsv', 'hetio_validation.tsv', 'hetio_test.tsv']:
    pre_df = pd.read_csv(df_name, sep='\t', header=None, names=['s', 'p','o'])
    pre_df['nice_p'] = pre_df.p.apply(lambda x: base_iri +property_map.get(x))
    pre_df['nice_s'] = pre_df.s.apply(lambda x: base_iri +x.split('::')[-1])
    pre_df['nice_o'] = pre_df.o.apply(lambda x: base_iri +x.split('::')[-1])
    pre_df[['nice_s','nice_p','nice_o']].to_csv('hetionet/'+ df_name.replace('.','_nice.'), sep='\t', index=False, header=False)

In [7]:
for f_name in ['hetio_train_graph.nt', 'hetio_full_graph.nt']:
    with open(f_name, 'r') as in_f:
        with open(f_name.replace('.','_noIRI.'), 'w') as out_f:
            for line in in_f:
                out_f.write(line.replace('<','').replace('>','') + '\n')

# Time to materialize

In [ ]:
tnice = pd.read_csv('hetio_train_nice.tsv', sep='\t', names = ['s', 'p','o'])

In [ ]:
tnice[tnice.p.str.contains('-')]

In [ ]:
property_map

In [27]:
emap = defaultdict(str)

with open('og_data_romy/entity_mapping.nt','r') as f:

    for line in f.readlines():
        names = line.replace('<','').replace('>','').split()
        emap['http://example.org/'+names[0]] = 'http://example.org/'+names[2]

In [28]:
emap

defaultdict(str,
            {'http://example.org/1': 'http://example.org/Gene',
             'http://example.org/10': 'http://example.org/Gene',
             'http://example.org/100': 'http://example.org/Gene',
             'http://example.org/1000': 'http://example.org/Gene',
             'http://example.org/10000': 'http://example.org/Gene',
             'http://example.org/10001': 'http://example.org/Gene',
             'http://example.org/10002': 'http://example.org/Gene',
             'http://example.org/10003': 'http://example.org/Gene',
             'http://example.org/100037417': 'http://example.org/Gene',
             'http://example.org/10004': 'http://example.org/Gene',
             'http://example.org/100049587': 'http://example.org/Gene',
             'http://example.org/10005': 'http://example.org/Gene',
             'http://example.org/10006': 'http://example.org/Gene',
             'http://example.org/10007': 'http://example.org/Gene',
             'http://example.org/

In [30]:
with open('hetionet/hetionet_entity_types.nt','w') as hn:
    for key,value in emap.items():
        hn.write('<' + key + '>\t<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>\t<' + value + '>\t.\n')

In [13]:
df = pd.read_csv('new_triples_False.txt', sep = ' ', header=None, names=['s', 'p','o','pkt'])[['s','p','o']]

In [14]:
df['subject_class'] = df['s'].apply(lambda x: emap.get(x,''))

In [17]:
df[df['subject_class'] != 'http://example.org/Anatomy']

,s,p,o,subject_class
0,2http://example.org/DB00014,http://example.org/expresses,http://example.org/6676,
1,http://example.org/DOID:10283,http://example.org/expresses,http://example.org/65110,http://example.org/Disease
2,http://example.org/DOID:10283,http://example.org/expresses,http://example.org/1477,http://example.org/Disease
3,http://example.org/DOID:10283,http://example.org/expresses,http://example.org/140739,http://example.org/Disease
4,http://example.org/DOID:10283,http://example.org/expresses,http://example.org/79544,http://example.org/Disease
...,...,...,...,...
111865,http://example.org/DOID:2377,http://example.org/expresses,http://example.org/126308,http://example.org/Disease
111866,http://example.org/DOID:2377,http://example.org/expresses,http://example.org/79004,http://example.org/Disease
111867,http://example.org/DOID:2377,http://example.org/expresses,http://example.org/84337,http://example.org/Disease
111868,http://example.org/DOID:2377,http://example.org/expresses,http://example.org/3975,http://example.org/Disease


In [18]:
pd.read_csv('hetio_test_nice.tsv').shape

(450035, 1)

In [8]:
newtest = pd.read_csv('hetio_train_nice.tsv', sep='\t', header=None, names=['s', 'p','o'])
reduced_df = newtest.groupby('p').head(3).to_csv('debug_hetio_test.tsv', header=False, index=False, sep='\t')

In [35]:
newtest[(newtest['s'] == 'http://example.org/DB01115') & (newtest['p'] == 'http://example.org/palliates')].head()

,s,p,o
49615,http://example.org/DB01115,http://example.org/palliates,http://example.org/DOID:418
240032,http://example.org/DB01115,http://example.org/palliates,http://example.org/DOID:3393


In [41]:
includes = newtest[newtest['p'] == 'http://example.org/includes']
obj_counts= includes.groupby('o')['s'].count()
obj_counts.value_counts()

s
1     536
2     120
3      27
4      17
5       4
6       3
7       2
8       1
13      1
10      1
Name: count, dtype: int64

In [42]:
newtest[newtest['p'] == 'http://example.org/includes']

,s,p,o
43913,http://example.org/N0000000069,http://example.org/includes,http://example.org/DB00270
43914,http://example.org/N0000000070,http://example.org/includes,http://example.org/DB00177
43915,http://example.org/N0000000083,http://example.org/includes,http://example.org/DB00471
43916,http://example.org/N0000000100,http://example.org/includes,http://example.org/DB00286
43917,http://example.org/N0000000102,http://example.org/includes,http://example.org/DB00285
...,...,...,...
2193516,http://example.org/N0000000233,http://example.org/includes,http://example.org/DB00993
2195110,http://example.org/N0000182141,http://example.org/includes,http://example.org/DB00701
2198649,http://example.org/N0000008832,http://example.org/includes,http://example.org/DB06209
2201650,http://example.org/N0000000190,http://example.org/includes,http://example.org/DB00950


In [43]:
filtered_and_sorted_df = (
    includes.groupby('o')
    .filter(lambda x: len(x) > 1) # Keep groups (objects) that have more than 1 subject
    .sort_values(by='o')         # Sort the final result by object ('o')
)

In [44]:
filtered_and_sorted_df

,s,p,o
44182,http://example.org/N0000175914,http://example.org/includes,http://example.org/DB00035
44183,http://example.org/N0000175915,http://example.org/includes,http://example.org/DB00035
43962,http://example.org/N0000006320,http://example.org/includes,http://example.org/DB00035
44112,http://example.org/N0000175084,http://example.org/includes,http://example.org/DB00050
44031,http://example.org/N0000008638,http://example.org/includes,http://example.org/DB00050
...,...,...,...
831497,http://example.org/N0000175459,http://example.org/includes,http://example.org/DB08934
1581424,http://example.org/N0000175976,http://example.org/includes,http://example.org/DB09009
62533,http://example.org/N0000007681,http://example.org/includes,http://example.org/DB09009
44054,http://example.org/N0000009371,http://example.org/includes,http://example.org/DB09020
